# Active vision tutorial

This notebook implements a compact active-vision pipeline for object perception.

The goal is to start from a static 3D object and generate a temporally structured sensorimotor sequence. We first render the object under small fixational camera movements. We then convert the RGB sequence into events, compute event-based proto-object attention, and extract regions of interest together with the corresponding attentional displacements.

The full pipeline is:

```text
3D object
→ drift-like fixational camera motion
→ RGB frame sequence
→ event stream
→ saliency map
→ temperature-weighted proto-object attention
→ ROIs and saccadic displacements
```

## 1. Setup

We first add the local `src/` directory to the Python path and check that Blender's Python API is available.

This notebook is designed to be run from the root of the repository.

In [ ]:
import sys
from pathlib import Path

repo = Path.cwd()
sys.path.insert(0, str(repo / "src"))

import bpy
print(bpy.app.version_string)

## 2. Select object and output paths

For this tutorial we use a single `.blend` object. All generated outputs are saved under `data/renders/`.

The same pipeline can be applied to other 3D objects by changing `object_path` and `output_root`.

In [ ]:
from scene.render_offline import (
    render_still,
    render_camera_motion_sequence,
    frames_to_gif,
    frames_to_events_npy_and_gif,
)

from IPython.display import Image, display

In [ ]:
object_path = repo / "data" / "airplane_010.blend"

output_root = repo / "data" / "renders" / "airplane_demo"
still_path = output_root / "still_test.png"
sequence_dir = output_root / "motion_sequence"
video_path = sequence_dir / "rgb_motion.mp4"

## 3. Render a still image

Before generating a temporal sequence, we render a single frame to verify that the object is correctly loaded, centred, scaled, illuminated, and visible from the camera.

The camera is first oriented to fixate the object. This gives us a stable reference pose before adding small drift-like perturbations.

In [ ]:
from scene.render_offline import render_still

result = render_still(
    object_path=object_path,
    output_path=still_path,
    resolution=256,
    samples=1,
    use_gpu=True,

    object_target_size=0.45,
    object_azimuth_deg=315.0,
    object_elevation_deg=30.0,

    camera_position=(0.0, -0.25, 2.0),
    camera_target=(0.0, 0.0, 0.0),
    focal_length=50.0,
    sensor_width_mm=32.0,

    light_location=(0.0, 0.0, 5.0),
    light_size=10.0,
    light_strength=50.0,
)

display(Image(filename=str(still_path)))

Now that we have rendered a static view of the object, try modifying the scene parameters and observe how the image changes.

The main parameters to explore are:

- `object_target_size` controls the apparent size of the object in the scene. Larger values make the object occupy more of the image.
- `object_azimuth_deg` rotates the object around the vertical axis, changing which side of the object is visible.
- `object_elevation_deg` tilts the object vertically, changing whether the object is viewed more from above or below.
- `camera_position` controls where the camera is placed in 3D space.
- `camera_target` defines where the camera is looking.
- `focal_length` changes the field of view. Larger focal lengths produce a more zoomed-in image, while smaller focal lengths give a wider view.
- `light_location`, `light_size`, and `light_strength` determine how the object is illuminated. Here, we use a rectangular Blender light.

Try changing one parameter at a time and re-running the cell. This makes it easier to identify which visual change is caused by each parameter.

As you experiment, ask:

1. Is the object clearly visible?
2. Is it centred in the image?
3. Does it occupy enough of the frame?
4. Is the viewpoint informative?
5. Is the lighting strong enough to reveal the shape of the object?
6. Would small camera movements produce meaningful visual changes?

A good initial render should be well framed, well illuminated, and contain enough visual structure.


## 4. Simulate drift-like fixational camera motion

Here we generate a short RGB sequence by adding small accumulated perturbations to the camera orientation.

Biological fixational eye movements include several components:

- **Drift**: slow continuous motion during fixation.
- **Microsaccades**: brief ballistic jumps that re-centre or shift gaze.
- **Tremor**: very small high-frequency oscillations.

In this implementation we model the **drift-like component**. The camera pan and tilt are updated as a Gaussian random walk:

$$
\theta^{pan}_t = \theta^{pan}_{t-1} + \epsilon^{pan}_t
$$

$$
\theta^{tilt}_t = \theta^{tilt}_{t-1} + \epsilon^{tilt}_t
$$

where the noise terms are sampled independently at each frame. Hence the generated motion is smooth and accumulated over time, rather than composed of discrete microsaccadic jumps.

In [ ]:
from scene.render_offline import render_camera_motion_sequence

seq = render_camera_motion_sequence(
    object_path=object_path,
    output_dir=sequence_dir,

    num_frames=500,
    fps=1000,

    resolution=256,
    samples=1,
    use_gpu=True,

    object_target_size=0.45,
    object_azimuth_deg=315.0,
    object_elevation_deg=30.0,

    camera_position=(0.0, -0.25, 2.0),
    camera_target=(0.0, 0.0, 0.0),
    focal_length=50.0,
    sensor_width_mm=32.0,

    light_location=(0.0, 0.0, 5.0),
    light_size=10.0,
    light_strength=50.0,

    drift_sigma_deg=(0.10, 0.09),

    seed=0,
)


In [ ]:
from scene.render_offline import frames_to_gif

gif_path = sequence_dir / "rgb_motion.gif"

gif_file = frames_to_gif(
    frames_dir=seq["frames_dir"],
    output_gif=gif_path,
    fps=30,
    loop=0,
)

display(Image(filename=str(gif_path)))

We now convert the RGB frame sequence into an event stream using the [IEBCS simulator](https://github.com/neuromorphicsystems/IEBCS). An event camera does not store full frames. Instead, it emits events when the log-intensity at a pixel changes sufficiently.

In [ ]:
from scene.render_offline import frames_to_events_npy_and_gif

ev = frames_to_events_npy_and_gif(
    frames_dir=seq["frames_dir"],
    output_dir=sequence_dir / "events",
    fps=1000,
    th_pos=0.15,
    th_neg=0.15,
    th_noise=0.05,
    lat=500,
    tau=300,
    jit=100,
    bgnp=0.001,
    bgnn=0.001,
    ref=40,
    skip_frames=0,
    gif_fps=20,
    gif_window_us=1000,
    loop=0,
)

display(Image(filename=ev["gif_path"]))

print("events.npy:", ev["npy_path"])
print("events.gif:", ev["gif_path"])
print("events.dat:", ev["dat_path"])

We have now rendered our DVS stream of events. Unlike a conventional RGB frame, this representation does not show the full visual scene at each time point. Instead, it shows where the image has changed over time.

In this example, events are generated because the camera is jittering. Even though the object itself is static, small camera movements shift the projected image across the sensor. This produces local changes in log-intensity, which are then converted into positive and negative events by the event-camera simulator.

This is an important point: **without motion, there should be little or no event activity**. Event cameras are not primarily measuring brightness itself. They are measuring brightness changes.

A similar principle applies to biological vision. If the image on the retina remains perfectly stable, retinal neurons adapt to the constant input and their response decreases over time. In other words, the visual system is highly sensitive to change, but much less responsive to unchanging stimulation. This is one reason why our eyes are never completely still. Even during fixation, the eyes produce tiny fixational eye movements, including drift, tremor, and microsaccades. These small movements continuously refresh the image on the retina and prevent stationary visual patterns from fading.

Try re-rendering the same object with no camera movement. What happens to the event stream?

You should observe that the number of events drops substantially, or may become almost empty, because the image projected onto the sensor is no longer changing. This provides a useful sanity check: the DVS stream is driven by visual change induced by motion, not by the static object alone.

- What happens if you increase the amplitude of the camera jitter?
- Do object edges generate more events than smooth regions?
- Are positive and negative events spatially balanced, or does one polarity dominate?
- What happens if the object is larger or closer to the camera?
- How does changing the lighting affect the number of events?

## 5. Event-based proto-object attention

Human vision is not a uniform camera-like process. When we look at a scene, we do not process the whole image with equal resolution. Instead, visual acuity is highest near the centre of the retina, in a small region called the fovea, where photoreceptor density is much higher. Because of this, humans actively move their eyes to bring informative parts of the scene onto the fovea.

This means that visual perception is an active process. Rather than passively receiving a full image, the visual system samples the scene through a sequence of fixations and eye movements. Larger saccades move gaze between relevant regions, while smaller fixational eye movements, such as drift and microsaccades, continuously perturb the retinal image even during fixation. These movements generate natural temporal variations of the same scene, creating biologically grounded augmentations that are paired with the eye movements that caused them.

Here, we use an event-based proto-object attention module to select informative regions from the event stream. The attention module processes events in short temporal windows. For each window, events are accumulated into a 2D event frame. A multiscale centre-surround operation is then applied to compute a saliency map. This approximates a proto-object attention mechanism, similar in spirit to the event-based attention approach used by [Giulia D'Angelo](https://github.com/GiuliaDAngelo/CTU-EDNeuromorphic/tree/main). The system does not yet recognise the object category, but it identifies spatially coherent regions that are likely to correspond to informative object parts. These regions can then be used to guide the next fixation. Then to introduce exploration over the object the selected fixation is sampled from a softmax distribution over the saliency map:

$$
p(x,y) = \frac{\exp(\beta S(x,y))}{\sum_{x',y'} \exp(\beta S(x',y'))}
$$

where $S(x,y)$ is the normalised saliency map and $\beta$ controls the sharpness of the distribution. Note that a larger $\beta$ makes attention more deterministic and concentrated around the most salient location, while a smaller $beta$ makes attention more exploratory, allowing the system to sample less salient regions as well.

In this way, the model links three components of active vision:

1. event-based visual input generated by small image changes,
2. proto-object saliency computed from local event structure,
3. foveated sampling guided by the selected fixation location.

The result is a simple biologically inspired attention loop: the system observes local temporal changes, estimates where informative structure is likely to be, and uses that estimate to decide where to look next.


In [ ]:
import numpy as np
from attention.attention import run_attention

attention_params = {
    "saliency_backend": "lif_vm",
    "num_pyr": 4,
    "lif_thetas": np.arange(0.0, 2.0 * np.pi, np.pi / 4),
    "lif_tau_mem": 0.3,
    "lif_size_krn": 16,
    "lif_rho": 0.1,
    "lif_r0": 14,
    "lif_thick": 3.0,
    "lif_offset": (0, 0),
    "lif_filter_resize_perc": 1.0,
    "lif_stride": 1,
    "lif_out_ch": 1,
    "lif_device": "auto",
    "lif_stateful": True,
    "beta": 10,
    "seed": 0,
}

att = run_attention(
    events_npy=ev["npy_path"],
    output_dir=sequence_dir / "out",
    resolution=(256, 256),
    window_period_ms=10.0,
    max_windows=None,
    sigma=None,
    use_polarity=False,
    clear_existing=True,
    attention_params=attention_params,
    plot=True,
    plot_gif_path=sequence_dir / "out" / "saliency_window_action.gif",
    plot_fps=10,
    plot_loop=0,
)

display(Image(filename=att["saliency_window_action_gif_path"]))

Nice! You now have a closed-loop active vision system with temperature-modulated proto-object attention.

The event stream is divided into short temporal windows. For each window, events are accumulated into an event frame, transformed into a saliency map, and used to sample the next fixation. The green arrow shows the selected displacement, indicating where the system decides to look next.

Try changing the parameters and observe how the behaviour changes.

- `window_period_ms` controls how much event activity is integrated before each attention update. Short windows make the system more reactive but noisier. Longer windows make saliency more stable but slower to update.
- `center_sigma` and `surround_sigma` control the centre-surround spatial scale. Smaller values emphasise local details, while larger values produce smoother, broader saliency maps.
- `beta` controls the attention temperature.

Ask yourself:

1. Does the arrow point towards regions with strong event activity?
2. Does the system explore different regions or keep returning to the same one?
3. Does increasing `beta` make the attention more deterministic?
4. Does increasing `window_period_ms` make the saliency map more stable?
5. Do the selected fixations correspond to meaningful object parts?

The key idea is that the system is no longer passively processing a fixed image. It uses event-driven visual changes to decide where to sample next.

Finally, we can quantify this behaviour by asking how much of the object was actually explored. The next plot compares the fixation trajectory with the object area and reports the fraction of object regions visited by the attention policy. This gives a simple readout of whether the system is narrowly exploiting a few salient parts or broadly sampling the object.

In [ ]:

from attention.attention import plot_attention_exploration

exploration = plot_attention_exploration(
    events_npy=ev["npy_path"],
    saccades_path=att["saccades_path"],
    output_dir=sequence_dir / "out",

    resolution=att["resolution"],
    beta=att["beta"],

    per=0.1,
    noise_thresh=100,

    exclude_initial_fixation=True,
    plot_trajectory=True,

    save_name="attention_exploration",
    save_png=True,
    save_pdf=True,
    show=True,
)

print(exploration)

This closed-loop behaviour also gives us a way to think about attention as a sampling process. If the sequence is too short, the system may generate only a few foveations and cover only a small part of the object. Rendering longer sequences, or allowing more attention windows, should produce more fixation samples and therefore broader object coverage.

The parameter `beta` is especially important because it controls the balance between exploration and exploitation. Low `beta` makes the policy more exploratory, while high `beta` makes it strongly attracted to the most salient regions. If `beta` is too high, the system can become distracted or trapped by a small number of highly salient event regions, reducing object coverage. Biologically, this is a simplified way to think about neuromodulatory control of attention. Neuromodulators such as acetylcholine and noradrenaline influence sensory gain, uncertainty, and attentional sampling, shaping how strongly the visual system prioritises salient information versus broader exploration of the scene [Thiele & Bellgrove, 2018](https://doi.org/10.1016/j.neuron.2018.01.008).


## 6. Foveation

Once the attention module has selected a fixation point, we use this point as the centre of a foveated view. Rather than cropping a square region from the image, this implementation applies a Gaussian spatial mask centred on the selected fixation. Pixels close to the fixation point are preserved, while pixels farther away are progressively suppressed.

For each attentional fixation $(x_t, y_t)$, we define a Gaussian foveation mask:

$$
G_t(x,y) =
\exp\left(
-\frac{1}{2}
\left[
\left(\frac{x - x_t}{\sigma}\right)^2
+
\left(\frac{y - y_t}{\sigma}\right)^2
\right]
\right)
$$

where $\sigma$ controls the spatial extent of the foveated region. Smaller values of $\sigma$ produce a narrower fovea and stronger peripheral suppression. Larger values preserve a broader region around the fixation point.

The foveated image is then obtained by multiplying the grayscale frame by the Gaussian mask:

$$
I_t^{\mathrm{fov}}(x,y) = I_t(x,y) \, G_t(x,y)
$$

This produces a sequence of full-frame foveated views:

$$
\{I_1^{\mathrm{fov}}, I_2^{\mathrm{fov}}, ..., I_T^{\mathrm{fov}}\}
$$

These foveated views show which parts of the object were emphasised by the attention system over time. The centre of each view corresponds to an attentional fixation, while the surrounding image is attenuated rather than removed. The resulting sequence provides local, fixation-centred visual samples while preserving the global frame of reference.

In [ ]:
from attention.foveation import foveations_to_gif

foveations_gif = foveations_to_gif(
    fov_dir=att["fov_dir"],
    output_gif=sequence_dir / "out" / "foveations.gif",
    fps=10,
    loop=0,
    image_prefix="roi",
)

display(Image(filename=str(foveations_gif)))

We have now generated a sequence of foveated views from the original object. Each foveation is centred on an attentional fixation point, and consecutive fixations define the corresponding saccadic displacement. This means that the dataset is no longer composed only of static object views, but of paired sensory and motor samples: local visual inputs together with the attentional action that links one view to the next. These foveation-saccade pairs can be used as structured augmentations for sensorimotor learning, especially in self-supervised settings where the model must learn object representations from temporally linked views rather than from explicit class labels.



---

This notebook accompanies the active-vision pipeline described in **Rodriguez-Garcia, A.**, Ghosh, A., Ramaswamy, S., & D'Angelo, G. (2025). *Object perception through visual attention*. Zenodo. https://doi.org/10.5281/zenodo.15802809